# Exercise 4: Chunking

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Meaning and Retrieval · version 0.1.0*

> Take the ten longest Kazakh articles in KazQAD and split them into 512-token chunks with three LLM tokenisers and one embedding tokeniser. How many words does a chunk hold on average with each? What does this mean for a RAG system's context budget?

**Setup.** `pip install torch transformers sentencepiece scipy bm25s requests "turkicnlp[hfst]==0.5.0"` (the compiled `hfst` package needs Python 3.11 or 3.12). Encoders are loaded at the Hugging Face revisions pinned in `MODELS`, from a local Hugging Face cache (`HF_HUB_DIR`), a folder of copies (`MODELS_DIR`) or the Hub; NLLB-200 is read from TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M`) as the `embeddings` processor does. Data come from commit-pinned GitHub repositories: SIB-200 (FLORES-200 sentences, the same 1,004 sentences in every language), Kardeş-NLU's STS sets and KazQAD's retrieval collection (825,309 Kazakh Wikipedia passages, about 230 MB compressed). Embeddings, lemma maps and retrieval runs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`). Everything runs on the CPU.

Articles are rebuilt by joining KazQAD's passages in order. Sentences are packed greedily into chunks of at most 512 tokens, counted without special tokens; a sentence longer than the limit is split at word boundaries.

In [ ]:
import collections, csv, gzip, hashlib, io, json, math, os, random, re, statistics, time
from pathlib import Path
import numpy as np, requests

SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KARDES = "https://raw.githubusercontent.com/lksenel/Kardes-NLU/48eea77fda904e731d228de833f4c57d130fad9c/Data/"
KAZQAD = "https://raw.githubusercontent.com/IS2AI/KazQAD/ab8684a667109f2a7345f4f89ab3b123e8d78e1f/data/information-retrieval/"
MODELS = {   # sentence encoder -> pinned revision (mean pooling, 128-token ceiling, as configured)
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
    "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a",
    "cis-lmu/glot500-base": "d4d7c1ec01828fdf7452a4ccf7b55177aced175e",                 # tokenizer only
    "meta-llama/Meta-Llama-3-8B": "8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920",          # tokenizer only
    "allenai/Olmo-3-1025-7B": "0a15c289b17ad4f27aabc863276ec525d5312c01",              # tokenizer only
    "Qwen/Qwen3.6-27B": "6a9e13bd6fc8f0983b9b99948120bc37f49c13e9",                    # tokenizer only
}
NAMES = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "MiniLM",
         "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "XLM-R para", "nllb": "NLLB enc"}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
HF_HUB_DIR = Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser()
MODELS_DIR = Path(os.environ.get("MODELS_DIR", CACHE / "models"))
NLLB_DIR = Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=600); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    FETCHED[url] = h.hexdigest()
    return path

def model_source(name):
    rev = MODELS[name]
    for p in (HF_HUB_DIR / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              MODELS_DIR / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev

def tokenizer(name):
    from transformers import AutoTokenizer
    path, rev = model_source(name)
    return AutoTokenizer.from_pretrained(path, revision=rev)

def sib(lang):
    """All 1,004 SIB-200 sentences of a FLORES language (train, dev, test), sorted by index_id."""
    rows = []
    for split in ("train", "dev", "test"):
        text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
        rows += [(int(r["index_id"]), r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]
    return [t for _, t in sorted(rows)]

TRANSLIT = {"kaz_Cyrl": ("kaz", "Cyrl"), "kir_Cyrl": ("kir", "Cyrl"), "tat_Cyrl": ("tat", "Cyrl"),
            "bak_Cyrl": ("bak", "Cyrl"), "uzn_Latn": ("uzb", "Latn"), "azj_Latn": ("aze", "Latn"), "tur_Latn": ("tur", "Latn")}
_T = {}
def to_cts(lang, text):
    """Common Turkic Alphabet version of a sentence with TurkicNLP's transliterator."""
    if lang not in _T:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        code, scr = TRANSLIT[lang]
        _T[lang] = Transliterator(code, Script.CYRILLIC if scr == "Cyrl" else Script.LATIN, Script.COMMON_TURKIC)
    return _T[lang].transliterate(text)

_ENC = {}
NLLB_CODE = {"kaz_Cyrl": "kaz_Cyrl", "kir_Cyrl": "kir_Cyrl", "tat_Cyrl": "tat_Cyrl", "bak_Cyrl": "bak_Cyrl",
             "uzn_Latn": "uzn_Latn", "azj_Latn": "azj_Latn", "tur_Latn": "tur_Latn", "rus_Cyrl": "rus_Cyrl",
             "eng_Latn": "eng_Latn", "uig_Arab": "uig_Arab"}
def embed(model, texts, lang=None, batch=32):
    """L2-normalised mean-pooled sentence vectors. Paraphrase encoders truncate at their configured
    128 tokens; the NLLB encoder (as in TurkicNLP's embeddings processor) gets the language token of
    `lang` and is not truncated below 512. Cached on disk by model, language and text hash."""
    key = hashlib.sha1(json.dumps([model, lang, texts], ensure_ascii=False).encode()).hexdigest()[:20]
    path = CACHE / "ch13-vectors" / f"{key}.npy"
    if path.exists():
        return np.load(path)
    import torch
    torch.set_num_threads(THREADS)
    if model not in _ENC:
        from transformers import AutoModel, AutoTokenizer
        if model == "nllb":
            from transformers import AutoModelForSeq2SeqLM
            tok = AutoTokenizer.from_pretrained(NLLB_DIR)
            enc = AutoModelForSeq2SeqLM.from_pretrained(NLLB_DIR).get_encoder().eval()
            _ENC[model] = (tok, enc, 512)
        else:
            src, rev = model_source(model)
            _ENC[model] = (AutoTokenizer.from_pretrained(src, revision=rev), AutoModel.from_pretrained(src, revision=rev).eval(), 128)
    tok, enc, limit = _ENC[model]
    if model == "nllb":
        tok.src_lang = NLLB_CODE[lang]
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], padding=True, truncation=True, max_length=limit, return_tensors="pt")
            h = enc(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            v = (h * m).sum(1) / m.sum(1).clamp(min=1)
            out.append(torch.nn.functional.normalize(v, dim=-1).numpy())
    vec = np.concatenate(out).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True); np.save(path, vec)
    return vec

def top1(q, c, targets):
    """Share of queries whose nearest candidate is the aligned one (exact duplicate target texts count)."""
    best = (q @ c.T).argmax(1)
    return float(np.mean([targets[b] == targets[i] for i, b in enumerate(best)]))

print("encoders:", ", ".join(f"{NAMES.get(n, n)} ({'local' if model_source(n)[1] is None else 'Hub'})" for n in list(MODELS)[:2]),
      f"+ NLLB encoder ({'local' if (NLLB_DIR / 'config.json').exists() else 'missing'})")

In [ ]:
CORPUS = [fetch(f"{KAZQAD}corpus/kazqad-corpus-v1.0-kk-part-{p}.jsonl.gz") for p in (1, 2, 3)]
articles = collections.defaultdict(list)
for p in CORPUS:
    with gzip.open(p, "rt", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line); parts = d["docid"].split("_")
            articles[parts[0]].append((tuple(int(x) for x in parts[1:]), d["title"], d["text"]))
def article_text(a):
    return "\n".join(t.strip() for _, _, t in sorted(articles[a]) if t.strip())
longest = sorted(articles, key=lambda a: -len(article_text(a).split()))[:10]
print("ten longest KazQAD articles (passages joined in order):")
for a in longest:
    print(f"  {articles[a][0][1][:40]:<40} {len(article_text(a).split()):>7,} words")

SENT = re.compile(r"(?<=[.!?])\s+|\n+")
TOKS = {"Llama 3": "meta-llama/Meta-Llama-3-8B", "OLMo 3": "allenai/Olmo-3-1025-7B", "Qwen3.6": "Qwen/Qwen3.6-27B",
        "Glot500 (embedding)": "cis-lmu/glot500-base"}
def chunks(text, tok, limit=512):
    """Greedy sentence packing: a chunk closes when the next sentence would exceed `limit` tokens.
    Sentences longer than the limit are split at word boundaries."""
    n = lambda s: len(tok.encode(s, add_special_tokens=False))
    units = []
    for s in (x.strip() for x in SENT.split(text)):
        if not s:
            continue
        if n(s) <= limit:
            units.append(s); continue
        cur = []
        for w in s.split():
            if cur and n(" ".join(cur + [w])) > limit:
                units.append(" ".join(cur)); cur = []
            cur.append(w)
        if cur: units.append(" ".join(cur))
    out, cur = [], ""
    for u in units:
        cand = (cur + " " + u).strip()
        if cur and n(cand) > limit:
            out.append(cur); cur = u
        else:
            cur = cand
    return out + ([cur] if cur else [])
print(f"\n512-token chunks, closed chunks only (the final tail of each article is excluded from the means):")
print(f"{'tokenizer':<20}{'chunks':>7}{'words/chunk':>13}{'tokens/word':>13}{'fill':>7}")
for name, model in TOKS.items():
    tok = tokenizer(model); nch = 0; words_, toks_ = [], []
    for a in longest:
        cs = chunks(article_text(a), tok); nch += len(cs)
        for c in cs[:-1]:
            words_.append(len(c.split())); toks_.append(len(tok.encode(c, add_special_tokens=False)))
    print(f"{name:<20}{nch:>7}{statistics.mean(words_):13.0f}{sum(toks_) / sum(words_):13.2f}{statistics.mean(toks_) / 512:7.1%}")

## Worked solution

Output of `experiments/ch13_exercises/run_ch13_exercises.py`:

```
ten longest KazQAD articles (passages joined in order):
  Ахмет Байтұрсынұлы атындағы тіл білімі и  43,030 words
  Қазақ мемлекеттігі                        27,091 words
  Қазақ ауыз әдебиеті мен араб әдебиеті ба  23,610 words
  Фридрих Ницше                             16,039 words
  Құдайберген Қуанұлы Жұбанов               15,352 words
  Қарағанды облысының жер-су аттары         14,993 words
  Аляска штатының қалалары                  14,672 words
  Философия және саясаттану факультеті (Қа  13,998 words
  Маңғыстау тарихы                          13,990 words
  Украина                                   13,744 words

512-token chunks, closed chunks only (the final tail of each article is excluded from the means):
tokenizer            chunks  words/chunk  tokens/word   fill
Llama 3                2065           95         4.90  91.2%
OLMo 3                 2592           76         6.01  89.1%
Qwen3.6                1564          126         3.80  93.7%
Glot500 (embedding)     972          203         2.38  94.4%
```

**Measured chunk sizes.** On the ten longest KazQAD articles, closed 512-token chunks average 95 whitespace words with Llama 3, 76 with OLMo 3, 126 with Qwen3.6 and 203 with Glot500. This sample differs from the FLORES probe in content and model selection. It does not isolate why fertility differs.

**Context budgets.** The same passage can have different token counts for the retriever and generator. Multiplying average words per chunk by average tokens per word gives only an approximation. Encode the selected passages together with the actual prompt and reserve response space; the means do not determine exactly how many passages will fit.

**Packing.** The algorithm greedily packs sentences, with mean fill of 89–94% here. Means exclude each article’s final tail. This measures packing, not retrieval accuracy or answer quality.

## Your turn — extension

1. Repeat with the Russian Wikipedia article on the same topic for two of the articles: how many words fit in a chunk?
2. Chunk by paragraph (KazQAD passages) instead of by sentence and compare the fill.